# Day10 — Kaggle setup · Văn Thành Huy
Notebook giai đoạn chuẩn bị: môi trường → dữ liệu chuẩn → minh họa → kiểm tra model → tải kết quả về local.
**Chưa triển khai vòng training E1–E4 trong phiên bản này.** Không sinh số liệu giả hoặc tự chạy các thí nghiệm tốn GPU.

Kaggle: **Import Notebook**, bật **Internet**, chọn **GPU T4 ×2**, Add Input dataset `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes`. Có thể setup dữ liệu trên CPU trước; bật GPU khi kiểm tra model.
Không cần đưa Kaggle key vào notebook khi dùng Add Input. Chạy từng cell, dừng nếu cell báo lỗi.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, zipfile
REPO_URL = "https://github.com/thanhhuyvan/K4-Track-4-Day10-VanThanhHuy-2A202602763--VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
# Có thể trỏ đến repo được upload dưới dạng Kaggle Input nếu GitHub chưa có mã đề.
REPO_INPUT = None
DATA_INPUT = None  # None: tìm tự động trong /kaggle/input; hoặc nhập đường dẫn dataset
LOAD_MODEL = False  # đổi True khi đã bật GPU và sẵn sàng tải model

def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=ROOT if ROOT.exists() else None)

if not ROOT.exists():
    if REPO_INPUT:
        shutil.copytree(REPO_INPUT, ROOT)
    else:
        run("git", "clone", REPO_URL, ROOT)
for required in ["scripts/prepare_data.py", "scripts/score.py", "requirements-colab.txt"]:
    assert (ROOT / required).is_file(), f"Repo thiếu {required}; upload repo vào Kaggle Input và đặt REPO_INPUT."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
(ROOT / "work").mkdir(exist_ok=True)
(ROOT / "report/evidence").mkdir(parents=True, exist_ok=True)
# Ghi hash để phát hiện vô tình sửa công cụ đề.
PROTECTED = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
             for folder in ["scripts", "tests"] for p in (ROOT / folder).rglob("*.py")}
print("Repo:", ROOT)


## 1. Cài dependencies
Giữ torch/torchvision của Kaggle. Cài đúng các phiên bản đề ghim; nếu pip báo lỗi, dừng và giữ log. Khi pip yêu cầu restart, restart session rồi chạy lại cell đầu.

In [ ]:
run(sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt", "pytest", "matplotlib")
run(sys.executable, "-m", "pip", "check")
run(sys.executable, "-m", "pytest", "-q")


In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes
runtime = {"python": sys.version, "torch": torch.__version__,
           "transformers": transformers.__version__, "peft": peft.__version__,
           "accelerate": accelerate.__version__, "bitsandbytes": bitsandbytes.__version__,
           "cuda_available": torch.cuda.is_available()}
if torch.cuda.is_available():
    runtime["gpus"] = [{"name": torch.cuda.get_device_name(i),
                        "vram_gb": torch.cuda.get_device_properties(i).total_memory / 2**30}
                       for i in range(torch.cuda.device_count())]
print(json.dumps(runtime, indent=2))
(ROOT / "work/runtime.json").write_text(json.dumps(runtime, indent=2))


## 2. Chuẩn bị dữ liệu bằng script gốc
Add Input dataset trước khi chạy. Không đổi quy tắc split của đề.

In [ ]:
from scripts.prepare_data import _find_root
base = Path(DATA_INPUT) if DATA_INPUT else Path("/kaggle/input")
try:
    source = _find_root(base)
except FileNotFoundError:
    raise RuntimeError("Chưa tìm thấy images/ và labels-YOLO/. Add Input dataset hoặc đặt DATA_INPUT đúng thư mục.")
run(sys.executable, "scripts/prepare_data.py", "--source", source)
meta = json.loads((ROOT / "data/splits/meta.json").read_text())
splits = {s: [json.loads(line) for line in (ROOT / f"data/splits/{s}.jsonl").read_text().splitlines() if line.strip()]
          for s in ["train", "val", "test"]}
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not sessions["train"] & sessions["val"]
assert not sessions["train"] & sessions["test"]
assert not sessions["val"] & sessions["test"]
print(json.dumps(meta["counts"], indent=2))
print("dataset_version:", meta["dataset_version"], "— không trùng phiên quay")


## 3. Chuyển nhãn và xem 6 ảnh train
Ảnh này chỉ minh họa ground truth, chưa phải ảnh dự đoán để nộp.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def objects_to_answer(objects):
    boxes = []
    for obj in objects:
        xc, yc, w, h = obj["yolo"]
        box = [max(0, min(1000, round(v * 1000))) for v in
               [xc-w/2, yc-h/2, xc+w/2, yc+h/2]]
        assert box[2] > box[0] and box[3] > box[1], "Box suy biến sau làm tròn"
        boxes.append({"label": obj["label"], "bbox_2d": box})
    boxes.sort(key=lambda x: (x["bbox_2d"][1], x["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

PROMPT = 'Locate all pothole, crack and manhole objects. Return only a JSON list of objects with label and bbox_2d=[x1,y1,x2,y2]. Use integer coordinates normalized to 0–1000 relative to image width and height. Return [] if none.'
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, row in zip(axes.flat, splits["train"][:6]):
    image = Image.open(ROOT / "data" / row["image_path"]).convert("RGB")
    ax.imshow(image)
    for box in json.loads(objects_to_answer(row["objects"])):
        x1,y1,x2,y2 = box["bbox_2d"]
        x1,x2 = x1*image.width/1000, x2*image.width/1000
        y1,y2 = y1*image.height/1000, y2*image.height/1000
        ax.add_patch(Rectangle((x1,y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
        ax.text(x1,y1,box["label"],color="white",backgroundcolor="black",fontsize=8)
    ax.set_title(row["image_id"],fontsize=8); ax.axis("off")
plt.tight_layout()
fig.savefig(ROOT / "work/train_ground_truth.png", dpi=140)
plt.show()
print(objects_to_answer(splits["train"][0]["objects"]))


## 4. Kiểm tra Qwen trên GPU (tùy chọn)
Đặt LOAD_MODEL=True ở cell đầu. Cell này tải model và kiểm tra 1 ảnh val, **không training, không chấm test**. Dùng GPU 0; hai T4 không tự gộp VRAM.

In [ ]:
if LOAD_MODEL:
    assert torch.cuda.is_available(), "Bật GPU trước khi nạp model"
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual", "lm_head"])
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID,
        quantization_config=quant, dtype=torch.float16, device_map={"": 0})
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = "left"
    model.eval()
    row = splits["val"][0]
    image = Image.open(ROOT / "data" / row["image_path"]).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                               {"type": "text", "text": PROMPT}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to("cuda:0")
    image_token_id = model.config.image_token_id
    count = int((inputs["input_ids"] == image_token_id).sum())
    print("Vision tokens:", count)
    assert count == 220, f"Đề yêu cầu 220 token; thực tế {count}. Kiểm tra processor trước training."
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    raw = processor.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    from scripts.score import parse_output
    print(raw)
    print("Parse hợp lệ:", parse_output(raw) is not None)
    (ROOT / "work/smoke_val.json").write_text(json.dumps({"image_id": row["image_id"], "raw": raw}, ensure_ascii=False))
else:
    print("Bỏ qua tải model. Setup dữ liệu có thể chạy trên CPU.")


## 5. Xuất gói mang về local
Chạy cell này sau mỗi giai đoạn. Gói kết quả gồm report, predictions/log nếu có và meta split; không chứa ảnh dữ liệu hoặc trọng số. Checkpoint được đóng gói riêng nếu bạn tạo work/checkpoints ở các giai đoạn training sau.

Trên Kaggle: Save Version để giữ Output, tải ZIP từ Output/Files. Giải nén gói kết quả vào repo local; không commit work hoặc checkpoints. REPORT.md vẫn cần hoàn thiện bằng kết quả thật.

In [ ]:
for rel, expected in PROTECTED.items():
    assert hashlib.sha256((ROOT / rel).read_bytes()).hexdigest() == expected, f"Đã sửa công cụ đề: {rel}"
archive = Path("/kaggle/working/day10_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [ROOT / "report", ROOT / "work"]:
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".json", ".jsonl", ".png", ".jpg", ".csv", ".txt", ".log"} and "checkpoints" not in p.relative_to(ROOT).parts:
                z.write(p, str(p.relative_to(ROOT)))
    if (ROOT / "data/splits/meta.json").exists():
        z.write(ROOT / "data/splits/meta.json", "data/splits/meta.json")
print("Kết quả:", archive, archive.stat().st_size, "bytes")
checkpoint_dir = ROOT / "work/checkpoints"
if checkpoint_dir.exists():
    print("Checkpoint:", shutil.make_archive("/kaggle/working/day10_checkpoints", "zip", checkpoint_dir))
from IPython.display import FileLink, display
display(FileLink(str(archive)))


## Tiếp theo
Sau khi setup và smoke test thành công: triển khai mask loss/collator → kiểm tra module LoRA → benchmark ngắn E1/E3 → E0 và training E1–E3 → E4 → ảnh dự đoán/báo cáo. Không dùng smoke_val.json làm evidence test. E5 tùy chọn chưa có trong notebook này.